## PLN 2026 - Aula 7
Prof. Maiko Spiess

---

## Representações densas e busca semântica

Neste notebook, vamos retomar a mesma base de descrições de jogos utilizada na Aula 6 e construir duas novas representações vetoriais:

1. **Word2Vec** — embeddings estáticos de palavras, treinados com o próprio corpus;
2. **BERTimbau** — representações contextuais obtidas com um modelo BERT pré-treinado para português.

Ao final, construiremos um sistema de **busca por similaridade**. Para uma mesma consulta livre, o sistema retornará os **3 textos mais semelhantes em cada espaço vetorial**, permitindo comparar os rankings produzidos pelas duas representações.

A lógica geral será:

```text
consulta livre
      ↓
representação da consulta
      ↓
similaridade com os documentos
      ↓
ranking
      ↓
3 textos mais próximos
```

A ideia central da aula é manter o mecanismo de busca praticamente constante e modificar a **forma de representar os textos**.

Assim, podemos observar como diferentes espaços vetoriais alteram aquilo que aparece como semanticamente semelhante.


In [3]:
# Instalar as bibliotecas que serão utilizadas nesta aula.
# No Google Colab, pandas, scikit-learn, numpy, nltk e torch
# normalmente já estão disponíveis.

%pip -q install gensim transformers


Note: you may need to restart the kernel to use updated packages.


## Carregando a base de dados

Utilizaremos a mesma base sintética da Aula 6.

Cada linha possui:

- um identificador (`id`);
- o nome do jogo (`animal`);
- sua categoria (`categoria`);
- uma descrição textual (`descricao`).

Nesta aula, a coluna `descricao` será representada em diferentes espaços vetoriais.

Selecione o arquivo CSV utilizado na aula anterior.


In [4]:
import ast
import pandas as pd

df = pd.read_csv("jogos_nlp_completo.csv")
df["tokens_sem_stopwords"] = df["tokens_sem_stopwords"].apply(
    ast.literal_eval
)

df = df.rename(
    columns={
        "Posicao": "id",
        "Nome": "animal",
        "Tags": "categoria",
    }
)

df["categoria"] = df["categoria"].apply(
    lambda x: str(x).split(",")[0].strip()
)
df["descricao"] = df["tokens_sem_stopwords"].apply(lambda t: " ".join(t))
df = df.reset_index(drop=True)

# Validação das colunas
colunas_esperadas = {"id", "animal", "categoria", "descricao"}
assert colunas_esperadas.issubset(df.columns)


ImportError: DLL load failed while importing strptime: An Application Control policy has blocked this file.

## Preparação dos textos para o Word2Vec

Na Aula 6, realizamos:

1. tokenização;
2. normalização;
3. remoção de *stopwords*.

Vamos reutilizar esse procedimento para o **Word2Vec**.

Neste caso, a preparação é útil porque o modelo trabalhará diretamente com sequências de palavras.

Para o **BERTimbau**, porém, utilizaremos o texto original. Modelos contextuais foram pré-treinados para processar sentenças completas e podem aproveitar informações presentes na pontuação, nas palavras funcionais e na ordem dos termos.

Portanto:

```text
Word2Vec → tokens normalizados e sem stopwords
BERT     → descrição original
```


In [ ]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
stopwords_en = set(stopwords.words("english"))

def tokenizar(texto):
    return texto.split()

def normalizar(tokens):
    tokens_normalizados = []
    for token in tokens:
        token = token.lower()
        token = re.sub(r"[^\w-]", "", token)
        if token:
            tokens_normalizados.append(token)
    return tokens_normalizados

def remover_stopwords(tokens):
    return [t for t in tokens if t not in stopwords_en]


## Representação 1 — Word2Vec

O **Word2Vec** aprende vetores de palavras a partir dos contextos em que elas aparecem.

Palavras que aparecem em contextos semelhantes tendem a ocupar regiões próximas do espaço vetorial. Diferentemente do Bag of Words e do TF-IDF, portanto, as dimensões do vetor não correspondem diretamente às palavras do vocabulário.

Nesta atividade, vamos treinar um pequeno modelo **Skip-gram** com o próprio corpus.

### Uma limitação importante

Nossa base possui apenas algumas dezenas de descrições. Isso é muito pouco para treinar um Word2Vec robusto. O resultado deve ser interpretado como uma **demonstração didática do mecanismo**, e não como um modelo semântico de alta qualidade.

Essa limitação também é instrutiva: o Word2Vec precisa aprender relações distribucionais a partir das ocorrências observadas no corpus.


In [ ]:
from gensim.models import Word2Vec

# Treinar Word2Vec no pequeno corpus da aula
modelo_w2v = Word2Vec(
    sentences=df["tokens_sem_stopwords"].tolist(),
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,          # Skip-gram
    seed=42,
    workers=1,     # ajuda na reprodutibilidade
    epochs=200
)

print("Tamanho do vocabulário:", len(modelo_w2v.wv))
print("Dimensões dos vetores:", modelo_w2v.vector_size)

# Algumas palavras disponíveis no vocabulário
list(modelo_w2v.wv.index_to_key)[:20]


Tamanho do vocabulário: 3175
Dimensões dos vetores: 100


['adventure',
 'shooter',
 'new',
 'players',
 'rpg',
 'world',
 'set',
 'gameplay',
 'player',
 'combat',
 'series',
 'features',
 'platform',
 'games',
 'indie',
 'action',
 'weapons',
 'puzzle',
 'enemies',
 'developed']

## Do vetor da palavra ao vetor do documento

O Word2Vec produz um vetor para cada **palavra**, mas nosso objetivo é comparar **descrições inteiras**.

Utilizaremos uma estratégia simples: calcular a média dos vetores das palavras presentes em cada descrição.

Se uma descrição contém os vetores:

$$
v_1, v_2, \dots, v_n
$$

o vetor do documento será:

$$
v_{doc} = \frac{1}{n}\sum_{i=1}^{n}v_i
$$

Essa estratégia perde informações sobre a ordem das palavras, mas permite construir uma representação única de cada texto.

In [ ]:
import numpy as np

def vetor_documento_word2vec(tokens):
    # Calcula a média dos vetores Word2Vec das palavras conhecidas.
    vetores = [
        modelo_w2v.wv[token]
        for token in tokens
        if token in modelo_w2v.wv
    ]

    if not vetores:
        return np.zeros(
            modelo_w2v.vector_size,
            dtype=np.float32
        )

    return np.mean(vetores, axis=0)


# Criar uma matriz: uma linha por descrição
embeddings_word2vec = np.vstack(
    df["tokens_sem_stopwords"].apply(
        vetor_documento_word2vec
    )
)

# DataFrame apenas para inspecionar a representação
df_word2vec = pd.DataFrame(
    embeddings_word2vec,
    columns=[
        f"w2v_{i:03d}"
        for i in range(
            embeddings_word2vec.shape[1]
        )
    ]
)

df_word2vec.insert(
    0,
    "id",
    df["id"].values
)

print(
    "Formato da matriz Word2Vec:",
    embeddings_word2vec.shape
)

# Mostrar somente algumas dimensões
df_word2vec.iloc[:5, :8]


Formato da matriz Word2Vec: (198, 100)


,id,w2v_000,w2v_001,w2v_002,w2v_003,w2v_004,w2v_005,w2v_006
0,1,0.131629,0.105191,-0.480848,0.260319,-0.163874,0.179082,0.219806
1,2,0.542374,-0.111358,-0.045712,0.103927,-0.408347,0.059395,0.369583
2,3,-0.018075,-0.046837,-0.269319,-0.012729,-0.369212,0.081554,0.093158
3,4,0.007322,-0.371800,-0.100583,0.360582,-0.192028,-0.039676,0.379266
4,5,0.101663,-0.107914,-0.003770,0.209697,-0.369333,0.333578,0.311365


## Representação 2 — BERTimbau

Agora utilizaremos o **BERTimbau Base**, um modelo BERT pré-treinado para português brasileiro.

A diferença conceitual em relação ao Word2Vec é importante:

- no Word2Vec, uma palavra possui essencialmente o mesmo vetor em todos os contextos;
- no BERT, a representação de cada token depende das outras palavras da sentença.

Assim, o modelo produz **representações contextuais**.

Usaremos o modelo:

```text
neuralmind/bert-base-portuguese-cased
```

O carregamento inicial exige o download do modelo. Em um ambiente Colab, uma GPU acelera a execução, mas não é obrigatória para esta base pequena.


In [ ]:
import torch

from transformers import AutoTokenizer, AutoModel

MODELO_BERT = "bert-base-uncased"

# Detectar automaticamente GPU, quando disponível
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo:", device)

# Carregar tokenizador e modelo
tokenizer_bert = AutoTokenizer.from_pretrained(
    MODELO_BERT,
    do_lower_case=True
)

modelo_bert = AutoModel.from_pretrained(
    MODELO_BERT
)

modelo_bert = modelo_bert.to(device)
modelo_bert.eval()

print("Modelo carregado:", MODELO_BERT)


OSError: [WinError 4551] An Application Control policy has blocked this file. Error loading "C:\Users\bielc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\lib\shm.dll" or one of its dependencies.

## Transformando uma sentença em um único vetor

O BERT retorna um vetor contextual para cada token.

Para comparar documentos inteiros, precisamos condensar esses vetores em uma única representação. Utilizaremos **mean pooling**:

1. o texto é tokenizado pelo BERT;
2. cada token recebe uma representação contextual;
3. calculamos a média das representações dos tokens válidos;
4. obtemos um único vetor para cada descrição.

Este procedimento é simples e transparente para fins didáticos.

É importante observar que o BERTimbau é um modelo BERT geral, e não um modelo especificamente ajustado para produzir *sentence embeddings*. O *pooling* que realizamos aqui é uma escolha metodológica para transformar seus estados contextuais em vetores de documentos.


In [ ]:
def mean_pooling(
    last_hidden_state,
    attention_mask
):
    # Calcula a média dos vetores dos tokens,
    # ignorando posições utilizadas apenas como padding.
    mascara = attention_mask.unsqueeze(-1).expand(
        last_hidden_state.size()
    ).float()

    soma = torch.sum(
        last_hidden_state * mascara,
        dim=1
    )

    quantidade = torch.clamp(
        mascara.sum(dim=1),
        min=1e-9
    )

    return soma / quantidade


def gerar_embeddings_bert(
    textos,
    batch_size=8
):
    # Gera um vetor por texto usando
    # BERTimbau + mean pooling.
    todos_embeddings = []

    for inicio in range(
        0,
        len(textos),
        batch_size
    ):
        lote = textos[
            inicio:inicio + batch_size
        ]

        entradas = tokenizer_bert(
            lote,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        entradas = {
            chave: valor.to(device)
            for chave, valor
            in entradas.items()
        }

        with torch.no_grad():
            saida = modelo_bert(**entradas)

        embeddings_lote = mean_pooling(
            saida.last_hidden_state,
            entradas["attention_mask"]
        )

        todos_embeddings.append(
            embeddings_lote.cpu().numpy()
        )

    return np.vstack(
        todos_embeddings
    )


# Usamos a descrição ORIGINAL
textos = df["descricao"].tolist()

embeddings_bert = gerar_embeddings_bert(
    textos
)

print(
    "Formato da matriz BERT:",
    embeddings_bert.shape
)


Formato da matriz BERT: (30, 768)


## Comparando as dimensões das representações

Agora temos duas matrizes, cada uma representando exatamente os mesmos documentos:

```text
embeddings_word2vec
embeddings_bert
```

As dimensões dos vetores são diferentes, mas isso não impede a comparação dentro de cada espaço.

O ponto central é que **não compararemos diretamente um vetor Word2Vec com um vetor BERT**. Cada consulta será transformada separadamente em cada espaço vetorial.


## Busca por similaridade

Para transformar os embeddings em um sistema de busca, utilizaremos novamente a **similaridade do cosseno**.

O procedimento será o mesmo para as três representações:

```text
consulta
   ↓
vetor da consulta
   ↓
similaridade com todos os documentos
   ↓
ordenar do mais semelhante para o menos semelhante
   ↓
selecionar os 3 primeiros
```

Manter o mesmo mecanismo de recuperação ajuda a isolar o efeito da **representação**.

A função abaixo recebe:

- o vetor de uma consulta;
- a matriz de embeddings dos documentos;
- o número `k` de resultados desejados.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar_top_k(
    vetor_consulta,
    matriz_documentos,
    k=3
):
    # Retorna os k documentos mais semelhantes
    # à consulta.
    vetor_consulta = np.asarray(
        vetor_consulta
    ).reshape(1, -1)

    similaridades = cosine_similarity(
        vetor_consulta,
        matriz_documentos
    )[0]

    indices = np.argsort(
        similaridades
    )[::-1][:k]

    resultado = df.iloc[
        indices
    ][
        [
            "id",
            "animal",
            "categoria",
            "descricao"
        ]
    ].copy()

    resultado.insert(
        0,
        "posicao",
        range(1, len(resultado) + 1)
    )

    resultado["similaridade"] = (
        similaridades[indices]
    )

    return resultado


## Representando a consulta com Word2Vec

Uma consulta Word2Vec precisa passar pela mesma preparação utilizada nos documentos:

```text
consulta
  ↓
tokenização
  ↓
normalização
  ↓
remoção de stopwords
  ↓
vetores das palavras conhecidas
  ↓
média
```

Há uma limitação adicional: palavras que nunca apareceram no pequeno corpus não existem no vocabulário do modelo. Elas são chamadas de **OOV (*out-of-vocabulary*)**.

A função abaixo também retorna quais palavras foram reconhecidas e quais ficaram fora do vocabulário.


In [ ]:
def representar_consulta_word2vec(
    texto
):
    tokens = tokenizar(texto)
    tokens = normalizar(tokens)
    tokens = remover_stopwords(tokens)

    conhecidos = [
        token
        for token in tokens
        if token in modelo_w2v.wv
    ]

    oov = [
        token
        for token in tokens
        if token not in modelo_w2v.wv
    ]

    if not conhecidos:
        return None, conhecidos, oov

    vetor = np.mean(
        [
            modelo_w2v.wv[token]
            for token in conhecidos
        ],
        axis=0
    )

    return vetor, conhecidos, oov


## Representando a consulta com BERTimbau

Para o BERT, utilizamos a **consulta original**.

A consulta passa pelo mesmo tokenizador e pelo mesmo procedimento de *mean pooling* empregado nas descrições da base.

Isso garante que consulta e documentos sejam representados no mesmo espaço vetorial.


In [ ]:
def representar_consulta_bert(
    texto
):
    return gerar_embeddings_bert(
        [texto]
    )[0]


## Sistema de busca comparativa

Agora podemos reunir as duas representações em uma única função.

Para uma mesma **consulta livre**, vamos:

1. criar sua representação Word2Vec;
2. criar sua representação BERTimbau;
3. calcular a similaridade do cosseno entre a consulta e todos os documentos em cada espaço;
4. retornar os **3 resultados mais próximos de cada representação**.

O procedimento de recuperação permanece constante:

```text
consulta
   ↓
embedding
   ↓
similaridade do cosseno
   ↓
ranking top-3
```

O que muda é o espaço de representação.

### Como interpretar a comparação

Os valores de similaridade são úteis para ordenar documentos **dentro de cada representação**.

Porém, não devemos concluir automaticamente que, por exemplo, `0.82` em um modelo é "melhor" que `0.74` em outro. Os escores são produzidos em espaços vetoriais diferentes e não estão necessariamente calibrados entre si.

O foco principal deve estar em:

- quais textos foram recuperados;
- em que ordem aparecem;
- se correspondem semanticamente à consulta;
- onde Word2Vec e BERTimbau concordam ou divergem.


## Executando as consultas

Agora podemos testar o sistema. Experimente digitar:

- "open world action adventure with guns"
- "fantasy role playing game with magic and dragons"
- "post-apocalyptic survival zombie crafting"
- "fast racing driving cars"


In [ ]:
def buscar_comparando(
    consulta,
    k=3
):
    print(f"\n{'='*50}\nConsulta: {consulta}\n{'='*50}")

    # 1. Representação Word2Vec
    vetor_w2v, conhecidos_w2v, oov_w2v = representar_consulta_word2vec(consulta)

    print("\n--- Word2Vec ---")
    print(f"Tokens conhecidos: {conhecidos_w2v if conhecidos_w2v else 'Nenhum'}")
    print(f"Tokens fora do vocabulário (OOV): {oov_w2v if oov_w2v else 'Nenhum'}")

    if vetor_w2v is not None:
        resultado_w2v = buscar_top_k(
            vetor_w2v,
            embeddings_word2vec,
            k=k
        )
        print("\nResultados Word2Vec:")
        print(resultado_w2v.to_markdown(index=False))
    else:
        print("\nNão foi possível representar a consulta em Word2Vec (todos os tokens são OOV).")
        resultado_w2v = None

    # 2. Representação BERTimbau
    vetor_bert = representar_consulta_bert(consulta)
    resultado_bert = buscar_top_k(
        vetor_bert,
        embeddings_bert,
        k=k
    )
    print("\n--- BERTimbau ---")
    print("Resultados BERTimbau:")
    print(resultado_bert.to_markdown(index=False))

    return {
        "word2vec": resultado_w2v,
        "bertimbau": resultado_bert
    }


consulta = input(
    "Digite sua consulta: "
).strip()

if not consulta:
    raise ValueError(
        "A consulta não pode estar vazia."
    )

resultado = buscar_comparando(
    consulta,
    k=5
)

Digite sua consulta: tem bico e asas

Consulta: tem bico e asas

--- Word2Vec ---
Tokens conhecidos: ['bico', 'asas']
Tokens fora do vocabulário (OOV): Nenhum

Resultados Word2Vec:
|   posicao |   id | animal   | categoria   | descricao                                                                                                                                 |   similaridade |
|----------:|-----:|:---------|:------------|:------------------------------------------------------------------------------------------------------------------------------------------|---------------:|
|         1 |   28 | tucano   | ave         | Animal coberto por penas, com asas e duas patas. Destaca-se pelo bico grande e colorido e se reproduz colocando ovos.                     |       0.968664 |
|         2 |   29 | galinha  | ave         | Tem o corpo coberto por penas, duas patas, asas curtas e bico. Passa a maior parte do tempo no solo e se reproduz por ovos.               |       0.951954 |
|      

## Extensão opcional — recuperando o TF-IDF da Aula 6

As duas representações densas são o foco desta aula, mas podemos manter o **TF-IDF** como uma linha de base (*baseline*).

Essa comparação permite observar a progressão:

```text
TF-IDF
  ↓
representação lexical ponderada

Word2Vec
  ↓
semântica distribucional estática

BERTimbau
  ↓
representação contextual
```

A célula seguinte cria novamente o TF-IDF e oferece uma função de busca compatível com a mesma lógica de recuperação.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Reconstruir o baseline TF-IDF da Aula 6
tfidf_vectorizer = TfidfVectorizer(
    analyzer=lambda tokens: tokens
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    df["tokens_sem_stopwords"]
)


def buscar_tfidf(
    consulta,
    k=3
):
    # Aplicar à consulta a mesma preparação
    # utilizada na Aula 6.
    tokens = tokenizar(consulta)
    tokens = normalizar(tokens)
    tokens = remover_stopwords(tokens)

    vetor_consulta = (
        tfidf_vectorizer.transform(
            [tokens]
        )
    )

    similaridades = cosine_similarity(
        vetor_consulta,
        tfidf_matrix
    )[0]

    indices = np.argsort(
        similaridades
    )[::-1][:k]

    resultado = df.iloc[
        indices
    ][
        [
            "id",
            "animal",
            "categoria",
            "descricao"
        ]
    ].copy()

    resultado.insert(
        0,
        "posicao",
        range(1, len(resultado) + 1)
    )

    resultado["similaridade"] = (
        similaridades[indices]
    )

    return resultado


# Exemplo:
# buscar_tfidf(
#     "animal que vive na água",
#     k=3
# )


## Conclusão

Nesta aula, mantivemos a mesma base e o mesmo princípio de comparação utilizado anteriormente, mas substituímos representações lexicais esparsas por dois tipos de **embeddings densos**.

Construímos:

- uma representação **Word2Vec**, aprendida a partir do próprio corpus;
- uma representação contextual com **BERTimbau + mean pooling**;
- um mecanismo comum de recuperação baseado em **similaridade do cosseno**;
- um sistema que retorna os **3 documentos mais próximos** em cada espaço.

A comparação mostra que uma busca semântica não depende apenas do algoritmo que ordena os resultados. Ela depende fortemente da maneira como textos e consultas são **representados**.

### Questões para discussão

1. Em quais consultas Word2Vec e BERTimbau recuperam os mesmos jogos?
2. Onde os rankings divergem?
3. Como o pequeno corpus afeta o Word2Vec?
4. O BERTimbau geral produz rankings semanticamente consistentes após o *mean pooling*?
5. Em quais casos o BERTimbau recupera relações que o Word2Vec não consegue identificar?
6. O que mudaria se o corpus tivesse milhares ou milhões de documentos?
7. Em um sistema maior, quando seria necessário substituir a busca exaustiva por um índice vetorial aproximado, como HNSW?
